# Data Cleaning, Validation & Processing
## Fluency-SpAM Experiment

This notebook processes raw jsPsych JSON data from a cognitive psychology experiment with two tasks:
- **VFT (Verbal Fluency Task)**: Participants type as many words as possible in a category within 3 minutes
- **SpAM (Spatial Arrangement Method)**: Participants arrange those words in 2D space based on perceived similarity

**Domains**: animals, foods, colours (11 subjects), body-parts (24 subjects), + furniture-practice  
**Subjects**: 35 total  
**Response languages**: English, Hindi (Devanagari), Romanized Hindi

## 1. Setup & Configuration

In [ ]:
import json
import re
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 50)

# ── Configuration ──────────────────────────────────────────────
KEEP_PRACTICE = False          # Set to True to include furniture-practice domain
DATA_PATH = "responses (1).json"

print("Setup complete.")

Setup complete.


In [2]:
# Install additional dependencies (run once)
# !pip install google-generativeai

## 2. Load & Parse JSON

Parse the nested jsPsych JSON into three flat DataFrames:
- **`df_vft`**: One row per word from the Verbal Fluency Task  
- **`df_spam`**: One row per word from the Spatial Arrangement Method  
- **`df_demo`**: One row per subject with demographics & survey responses

In [3]:
# ── Load raw JSON ──────────────────────────────────────────────
with open(DATA_PATH, 'r', encoding='utf-8') as f:
    raw = json.load(f)

subjects = raw['fluency-spam']
print(f"Loaded {len(subjects)} subjects")

Loaded 35 subjects


In [4]:
# ── Parse VFT trials ───────────────────────────────────────────
vft_rows = []

for session_id, subj in subjects.items():
    subject_id = subj['subject_id']
    for trial in subj['data']:
        if trial.get('task') != 'VFT':
            continue
        domain = trial.get('domain', '')
        
        # tagged_responses is a JSON string: [{"response": "word", "tag": 1}, ...]
        tagged_str = trial.get('tagged_responses', '[]')
        tagged = json.loads(tagged_str) if isinstance(tagged_str, str) else tagged_str
        
        # response_times is a JSON string: [rt1, rt2, ...]
        rt_str = trial.get('response_times', '[]')
        rts = json.loads(rt_str) if isinstance(rt_str, str) else rt_str
        
        for i, entry in enumerate(tagged):
            word = entry.get('response', '').strip()
            tag = entry.get('tag', i + 1)
            rt = rts[i] if i < len(rts) else None
            
            vft_rows.append({
                'subject_id': subject_id,
                'session_id': session_id,
                'domain': domain,
                'word': word,
                'word_order': tag,
                'response_time_ms': rt,
            })

df_vft = pd.DataFrame(vft_rows)
print(f"VFT table: {df_vft.shape[0]} rows × {df_vft.shape[1]} columns")
print(f"Domains: {df_vft['domain'].unique().tolist()}")
print(f"Subjects: {df_vft['subject_id'].nunique()}")
df_vft.head(10)

VFT table: 1232 rows × 6 columns
Domains: ['furniture-practice', 'colours', 'animals', 'foods', 'body-parts']
Subjects: 35


,subject_id,session_id,domain,word,word_order,response_time_ms
0,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,chair,1,9613.2
1,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,sofa,2,1365.9
2,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,bench,3,1445.4
3,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,lights,4,3981.5
4,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,dinig table,5,2808.5
5,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,wadrobe,6,8129.8
6,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,mirror,7,3742.8
7,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,bed,8,5691.1
8,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,Tv stand,9,11889.2
9,10255,1qmxoH7jT7VECeLUVEKU,colours,red,1,2558.3


In [5]:
# ── Parse SpAM trials ──────────────────────────────────────────
spam_rows = []

for session_id, subj in subjects.items():
    subject_id = subj['subject_id']
    for trial in subj['data']:
        if trial.get('task') != 'SpAM':
            continue
        domain = trial.get('domain', '')
        droppedwords = trial.get('droppedwords', [])
        
        for entry in droppedwords:
            spam_rows.append({
                'subject_id': subject_id,
                'session_id': session_id,
                'domain': domain,
                'word': entry.get('word', '').strip(),
                'word_id': entry.get('id', ''),
                'x_px': entry.get('x_px'),
                'y_px': entry.get('y_px'),
                'x_norm': entry.get('x_norm'),
                'y_norm': entry.get('y_norm'),
                'canvas_width': entry.get('canvas_width'),
                'canvas_height': entry.get('canvas_height'),
            })

df_spam = pd.DataFrame(spam_rows)
print(f"SpAM table: {df_spam.shape[0]} rows × {df_spam.shape[1]} columns")
print(f"Domains: {df_spam['domain'].unique().tolist()}")
print(f"Subjects: {df_spam['subject_id'].nunique()}")
df_spam.head(10)

SpAM table: 1959 rows × 11 columns
Domains: ['furniture-practice', 'colours', 'animals', 'foods', 'body-parts']
Subjects: 35


,subject_id,session_id,domain,word,word_id,x_px,y_px,x_norm,y_norm,canvas_width,canvas_height
0,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,chair,word-1,702.200035,518.800003,0.574602,1.003870,1272.0,543.200012
1,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,sofa,word-2,611.200035,515.200027,0.498202,0.996904,1272.0,543.200012
2,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,bench,word-3,424.200035,511.200027,0.349398,0.989164,1272.0,543.200012
3,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,lights,word-4,1198.199974,337.200027,0.982554,0.652477,1272.0,543.200012
4,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,dinig table,word-5,800.200035,518.800003,0.674591,1.003870,1272.0,543.200012
5,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,chair,word-1,808.800011,480.399979,0.661832,0.929566,1272.0,543.200012
6,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,wadrobe,word-6,243.200005,439.200027,0.203238,0.849845,1272.0,543.200012
7,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,mirror,word-7,250.200005,411.200027,0.206382,0.795666,1272.0,543.200012
8,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,bed,word-8,255.200005,518.800003,0.207581,1.003870,1272.0,543.200012
9,10255,1qmxoH7jT7VECeLUVEKU,furniture-practice,Tv stand,word-9,600.200035,460.200027,0.500391,0.890480,1272.0,543.200012


In [6]:
# ── Parse Demographics / Survey trials ─────────────────────────
demo_rows = {}

for session_id, subj in subjects.items():
    subject_id = subj['subject_id']
    record = {
        'subject_id': subject_id,
        'session_id': session_id,
        'created_at': subj.get('created_at', ''),
        'user_agent': subj.get('user_agent', ''),
    }
    
    # Determine which domains this subject had (condition)
    subj_domains = set()
    for trial in subj['data']:
        d = trial.get('domain', '')
        if d and d != 'furniture-practice':
            subj_domains.add(d)
    record['domains'] = ', '.join(sorted(subj_domains))
    
    # Extract survey responses
    for trial in subj['data']:
        tt = trial.get('trial_type', '')
        resp = trial.get('response', {})
        
        if 'survey' not in tt:
            continue
        if not isinstance(resp, dict):
            continue
        
        # Merge all survey fields into the record
        for key, val in resp.items():
            # Avoid overwriting subject_id etc.
            if key not in ('subject_id', 'session_id'):
                record[key] = val
    
    demo_rows[subject_id] = record

df_demo = pd.DataFrame(demo_rows.values())
print(f"Demographics table: {df_demo.shape[0]} rows × {df_demo.shape[1]} columns")
print(f"\nColumns: {df_demo.columns.tolist()}")
df_demo.head()

Demographics table: 35 rows × 38 columns

Columns: ['subject_id', 'session_id', 'created_at', 'user_agent', 'domains', 'strategies', 'Hi_Read', 'Hi_Write', 'En_Read', 'En_Write', 'first_language', 'language_count', 'languages_buffer', 'confidence_0', 'confidence_1', 'confidence_2', 'acquisition_0', 'acquisition_1', 'acquisition_2', 'state_ut', 'gender', 'age', 'education', 'dominant_hand', 'alert_time', 'additional_info', 'confidence_3', 'confidence_4', 'acquisition_3', 'acquisition_4', 'confidence_5', 'confidence_6', 'confidence_7', 'confidence_8', 'acquisition_5', 'acquisition_6', 'acquisition_7', 'acquisition_8']


,subject_id,session_id,created_at,user_agent,domains,strategies,Hi_Read,Hi_Write,En_Read,En_Write,first_language,language_count,languages_buffer,confidence_0,confidence_1,confidence_2,acquisition_0,acquisition_1,acquisition_2,state_ut,gender,age,education,dominant_hand,alert_time,additional_info,confidence_3,confidence_4,acquisition_3,acquisition_4,confidence_5,confidence_6,confidence_7,confidence_8,acquisition_5,acquisition_6,acquisition_7,acquisition_8
0,10255,1qmxoH7jT7VECeLUVEKU,"{'_seconds': 1770295025, '_nanoseconds': 94200...",Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,"animals, colours, foods",,4,4,5,5,Telugu,3,Telugu||Hindi||English,5,4,5,1,2,2,Andhra Pradesh,Male,27,20,Right,Morning,no,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,95712,2kupFqWyVgUTsqisMsI7,"{'_seconds': 1770295884, '_nanoseconds': 50700...",Mozilla/5.0 (X11; Ubuntu; Linux x86_64; rv:147...,"animals, colours, foods",it's based on my liking or based on the sequen...,5,4,5,4,Hindi,2,Hindi||English,5,4,NaN,1,2,NaN,Chhattisgarh,Male,23,17,Right,No difference,i am used to type in english but don't have th...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,53105,32zXoOEyomBdbyGIOg7U,"{'_seconds': 1770628963, '_nanoseconds': 99900...",Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,"animals, body-parts, foods",,5,5,5,5,Gujarati,3,Gujarati||Hindi||English,5,5,4,1,2,3,Gujarat,Male,21,16,Right,Morning,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,53307,3zIUF3kkspNbB0QxKfOy,"{'_seconds': 1770660940, '_nanoseconds': 82500...",Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,"animals, body-parts, foods",,5,3,5,5,Punjabi,3,English||Hindi||Punjabi,4,4,5,3,2,1,Punjab,Male,21,16,Left,Evening,,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,92821,7fxWA2O6NjFEe7m2ElN6,"{'_seconds': 1770302678, '_nanoseconds': 29000...",Mozilla/5.0 (Windows NT 10.0; Win64; x64) Appl...,"animals, colours, foods",Memorization,4,4,4,4,Hindi,2,English||Hindi,4,4,NaN,1,2,NaN,Uttarakhand,Male,25,18,Right,Morning,N/A,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## 3. Basic Cleaning

- Remove rows with empty/null words
- Apply `KEEP_PRACTICE` toggle to filter out furniture-practice domain
- Report shape before and after

In [7]:
# ── Remove empty / null words ──────────────────────────────────
print("=== VFT Cleaning ===")
print(f"Before: {len(df_vft)} rows")

# Remove rows where word is empty, null, or whitespace-only
df_vft = df_vft[df_vft['word'].notna() & (df_vft['word'].str.strip() != '')]
df_vft['word'] = df_vft['word'].str.strip()
print(f"After removing empty words: {len(df_vft)} rows")

# Apply practice domain toggle
if not KEEP_PRACTICE:
    df_vft = df_vft[df_vft['domain'] != 'furniture-practice'].copy()
    print(f"After removing practice domain: {len(df_vft)} rows")

print(f"\nDomain distribution:\n{df_vft['domain'].value_counts()}")

print("\n=== SpAM Cleaning ===")
print(f"Before: {len(df_spam)} rows")

df_spam = df_spam[df_spam['word'].notna() & (df_spam['word'].str.strip() != '')]
df_spam['word'] = df_spam['word'].str.strip()
print(f"After removing empty words: {len(df_spam)} rows")

if not KEEP_PRACTICE:
    df_spam = df_spam[df_spam['domain'] != 'furniture-practice'].copy()
    print(f"After removing practice domain: {len(df_spam)} rows")

print(f"\nDomain distribution:\n{df_spam['domain'].value_counts()}")

=== VFT Cleaning ===
Before: 1232 rows
After removing empty words: 1232 rows
After removing practice domain: 1044 rows

Domain distribution:
domain
animals       365
foods         326
body-parts    206
colours       147
Name: count, dtype: int64

=== SpAM Cleaning ===
Before: 1959 rows
After removing empty words: 1959 rows
After removing practice domain: 1661 rows

Domain distribution:
domain
animals       581
foods         506
body-parts    337
colours       237
Name: count, dtype: int64


## 4. Dropout Check

Verify every subject completed all 3 real domains for both VFT and SpAM.

In [8]:
# ── Check for incomplete subjects ──────────────────────────────
# Each subject should have exactly 3 real domains in both VFT and SpAM

vft_domains_per_subject = df_vft.groupby('subject_id')['domain'].apply(set)
spam_domains_per_subject = df_spam.groupby('subject_id')['domain'].apply(set)

# Expected: each subject has 3 domains
incomplete_vft = vft_domains_per_subject[vft_domains_per_subject.apply(len) < 3]
incomplete_spam = spam_domains_per_subject[spam_domains_per_subject.apply(len) < 3]

if len(incomplete_vft) == 0 and len(incomplete_spam) == 0:
    print("✓ All subjects completed all 3 real domains in both VFT and SpAM.")
else:
    if len(incomplete_vft) > 0:
        print(f"⚠ {len(incomplete_vft)} subjects with incomplete VFT:")
        print(incomplete_vft)
    if len(incomplete_spam) > 0:
        print(f"⚠ {len(incomplete_spam)} subjects with incomplete SpAM:")
        print(incomplete_spam)

# Word count summary per subject × domain
vft_counts = df_vft.groupby(['subject_id', 'domain']).size().reset_index(name='word_count')
print(f"\nVFT word count stats per (subject, domain):")
print(vft_counts['word_count'].describe().round(1))

print(f"\nVFT word counts by domain:")
print(vft_counts.groupby('domain')['word_count'].describe().round(1))

✓ All subjects completed all 3 real domains in both VFT and SpAM.

VFT word count stats per (subject, domain):
count    105.0
mean       9.9
std        4.2
min        3.0
25%        7.0
50%        9.0
75%       12.0
max       24.0
Name: word_count, dtype: float64

VFT word counts by domain:
            count  mean  std  min   25%   50%   75%   max
domain                                                   
animals      35.0  10.4  4.6  4.0   7.0  10.0  12.0  23.0
body-parts   24.0   8.6  3.1  4.0   6.0   8.0  10.2  16.0
colours      11.0  13.4  3.4  7.0  12.0  14.0  15.0  19.0
foods        35.0   9.3  4.1  3.0   7.0   8.0  10.5  24.0


## 6. Duplicate Detection

Flag duplicate words within the same (subject, domain) pair. The first occurrence is kept as non-duplicate.

In [9]:
# ── Flag duplicate words within (subject, domain) ─────────────

# Normalize for comparison: lowercase, strip whitespace
df_vft['word_lower'] = df_vft['word'].str.lower().str.strip()

# Flag duplicates — keep='first' marks the 2nd+ occurrence as True
df_vft['is_duplicate'] = df_vft.duplicated(subset=['subject_id', 'domain', 'word_lower'], keep='first')

n_dupes = df_vft['is_duplicate'].sum()
print(f"Duplicate words found: {n_dupes} out of {len(df_vft)} ({n_dupes/len(df_vft)*100:.1f}%)")

if n_dupes > 0:
    print("\nDuplicate entries:")
    dupes = df_vft[df_vft['is_duplicate']][['subject_id', 'domain', 'word', 'word_order']]
    display(dupes)

# Also check SpAM for duplicates
df_spam['word_lower'] = df_spam['word'].str.lower().str.strip()
df_spam['is_duplicate'] = df_spam.duplicated(subset=['subject_id', 'domain', 'word_lower'], keep='first')
spam_dupes = df_spam['is_duplicate'].sum()
print(f"\nSpAM duplicates: {spam_dupes} out of {len(df_spam)}")

Duplicate words found: 7 out of 1044 (0.7%)

Duplicate entries:


,subject_id,domain,word,word_order
69,95712,colours,नीला,6
462,68981,body-parts,सिर,2
503,61476,colours,peela,9
504,61476,colours,hara,10
573,62335,animals,चीता,8
830,81788,colours,orange,9
875,34350,colours,pink,8



SpAM duplicates: 628 out of 1661


In [ ]:
# ── Apply API results to df_vft and df_spam ───────────────────

API_CSV = 'api_results.csv'
DEV_PATTERN = re.compile(r'[\u0900-\u097F]')

df_api = pd.read_csv(API_CSV)
print(f"✓ Loaded {len(df_api)} rows from {API_CSV}")

# ── 1. Merge API columns onto VFT and SpAM ───────────────────
merge_cols_vft  = ['domain', 'word', 'script', 'word_normalized', 'in_category']
merge_cols_spam = ['domain', 'word', 'script', 'word_normalized']

for col in ['script', 'word_normalized', 'in_category']:
    df_vft  = df_vft.drop(columns=[col],  errors='ignore')
    df_spam = df_spam.drop(columns=[col], errors='ignore')

df_vft  = df_vft.merge(df_api[merge_cols_vft],  on=['domain', 'word'], how='left')
df_spam = df_spam.merge(df_api[merge_cols_spam], on=['domain', 'word'], how='left')

# ── 2. Deterministic script override ─────────────────────────
# Word contains Devanagari chars → script must be 'devanagari',
# regardless of what the CSV says (safety net for stale CSVs).
for label, df in [('VFT', df_vft), ('SpAM', df_spam)]:
    dev_mask = df['word'].str.contains(r'[\u0900-\u097F]', regex=True, na=False)
    n_fixed = (dev_mask & (df['script'] != 'devanagari')).sum()
    df.loc[dev_mask, 'script'] = 'devanagari'
    if n_fixed > 0:
        print(f"  ⚠ {label}: fixed {n_fixed} Devanagari words with wrong script")

# ── 3. Fill unmatched words (not in API CSV) ──────────────────
for label, df in [('VFT', df_vft), ('SpAM', df_spam)]:
    na_mask = df['script'].isna()
    if na_mask.any():
        is_dev = df['word'].str.contains(r'[\u0900-\u097F]', regex=True, na=False)
        df.loc[na_mask &  is_dev, 'script'] = 'devanagari'
        df.loc[na_mask & ~is_dev, 'script'] = 'unknown'
        df['word_normalized'] = df['word_normalized'].fillna(df['word'].str.lower())
        print(f"  ⚠ {label}: {na_mask.sum()} words not in API CSV, filled with fallback")

# ── 4. Mark English words invalid (Hindi fluency test) ────────
eng_mask = df_vft['script'] == 'english'
df_vft.loc[eng_mask, 'in_category'] = False
print(f"\n⚠ {eng_mask.sum()} English words marked invalid (Hindi fluency test)")

# ── 5. Results ────────────────────────────────────────────────
print("\n── Script Distribution (VFT) ──")
print(df_vft['script'].value_counts().to_string())

# Transliterated Devanagari sample
dev = df_vft[df_vft['script'] == 'devanagari'][['domain', 'word', 'word_normalized']].drop_duplicates()
print(f"\nDevanagari → Latin ({len(dev)} unique words)")
display(dev.head(20))

# Flag any Devanagari words still missing transliteration
still_dev = df_vft[
    (df_vft['script'] == 'devanagari') &
    df_vft['word_normalized'].str.contains(r'[\u0900-\u097F]', regex=True, na=False)
][['domain', 'word', 'word_normalized']].drop_duplicates()
if len(still_dev) > 0:
    print(f"\n⚠ {len(still_dev)} Devanagari words still missing Latin transliteration:")
    display(still_dev)
else:
    print("✓ All Devanagari words have Latin transliteration.")

# Unification check: different original forms → same normalized form
unified = df_vft.groupby(['domain', 'word_normalized'])['word'].apply(
    lambda g: list(g.unique())
).reset_index(name='original_forms')
multi = unified[unified['original_forms'].apply(len) > 1]
if len(multi) > 0:
    print(f"\n── Words unified across scripts ({len(multi)}) ──")
    display(multi)

# Invalid words summary
flagged = df_vft[df_vft['in_category'] == False]
if len(flagged) > 0:
    n_ood = ((df_vft['in_category'] == False) & (df_vft['script'] != 'english')).sum()
    print(f"\n── Invalid Words: {n_ood} out-of-category + {eng_mask.sum()} English ──")
    display(flagged[['subject_id', 'domain', 'word', 'word_normalized', 'script']].drop_duplicates())
else:
    print("\n✓ All words valid.")

## 9. Cleaning Summary & Export

Final overview of the cleaned data and export to CSV files.

In [13]:
# ── Cleaning Summary ───────────────────────────────────────────

print("=" * 60)
print("DATA CLEANING SUMMARY")
print("=" * 60)

print(f"\n📊 VFT DataFrame: {df_vft.shape[0]} rows × {df_vft.shape[1]} columns")
print(f"   Subjects: {df_vft['subject_id'].nunique()}")
print(f"   Domains:  {df_vft['domain'].unique().tolist()}")
if 'script' in df_vft.columns:
    print(f"   Script distribution:")
    for script, count in df_vft['script'].value_counts().items():
        pct = count / len(df_vft) * 100
        print(f"     {script}: {count} ({pct:.1f}%)")
print(f"   Duplicates flagged: {df_vft['is_duplicate'].sum()}")
if 'in_category' in df_vft.columns:
    n_invalid = (df_vft['in_category'] == False).sum()
    n_unchecked = df_vft['in_category'].isna().sum()
    print(f"   Out-of-category:    {n_invalid}")
    print(f"   Unchecked:          {n_unchecked}")

print(f"\n📊 SpAM DataFrame: {df_spam.shape[0]} rows × {df_spam.shape[1]} columns")
print(f"   Subjects: {df_spam['subject_id'].nunique()}")
print(f"   Domains:  {df_spam['domain'].unique().tolist()}")

print(f"\n📊 Demographics DataFrame: {df_demo.shape[0]} rows × {df_demo.shape[1]} columns")
print(f"   Columns: {df_demo.columns.tolist()}")

print(f"\n📋 VFT Columns: {df_vft.columns.tolist()}")
print(f"📋 SpAM Columns: {df_spam.columns.tolist()}")

DATA CLEANING SUMMARY

📊 VFT DataFrame: 1044 rows × 11 columns
   Subjects: 35
   Domains:  ['colours', 'animals', 'foods', 'body-parts']
   Script distribution:
     devanagari: 431 (41.3%)
     english: 335 (32.1%)
     romanized_hindi: 278 (26.6%)
   Duplicates flagged: 7
   Out-of-category:    338
   Unchecked:          0

📊 SpAM DataFrame: 1661 rows × 15 columns
   Subjects: 35
   Domains:  ['colours', 'animals', 'foods', 'body-parts']

📊 Demographics DataFrame: 35 rows × 38 columns
   Columns: ['subject_id', 'session_id', 'created_at', 'user_agent', 'domains', 'strategies', 'Hi_Read', 'Hi_Write', 'En_Read', 'En_Write', 'first_language', 'language_count', 'languages_buffer', 'confidence_0', 'confidence_1', 'confidence_2', 'acquisition_0', 'acquisition_1', 'acquisition_2', 'state_ut', 'gender', 'age', 'education', 'dominant_hand', 'alert_time', 'additional_info', 'confidence_3', 'confidence_4', 'acquisition_3', 'acquisition_4', 'confidence_5', 'confidence_6', 'confidence_7', 'confi

In [14]:
# ── Export to CSV ──────────────────────────────────────────────

# Select columns for export
vft_export_cols = ['subject_id', 'domain', 'word', 'word_normalized', 'word_order',
                   'response_time_ms', 'script', 'is_duplicate']
if 'in_category' in df_vft.columns:
    vft_export_cols.append('in_category')

spam_export_cols = ['subject_id', 'domain', 'word', 'word_normalized', 'word_id',
                    'x_px', 'y_px', 'x_norm', 'y_norm', 'canvas_width', 'canvas_height',
                    'script', 'is_duplicate']

df_vft[vft_export_cols].to_csv('vft_clean.csv', index=False, encoding='utf-8-sig')
df_spam[spam_export_cols].to_csv('spam_clean.csv', index=False, encoding='utf-8-sig')
df_demo.to_csv('demographics.csv', index=False, encoding='utf-8-sig')

print("✓ Exported:")
print(f"  vft_clean.csv     ({len(df_vft)} rows)")
print(f"  spam_clean.csv    ({len(df_spam)} rows)")
print(f"  demographics.csv  ({len(df_demo)} rows)")

✓ Exported:
  vft_clean.csv     (1044 rows)
  spam_clean.csv    (1661 rows)
  demographics.csv  (35 rows)
